In [1]:
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 10.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 10.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 8.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.0/814.0 kB 13.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 10.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [mlflow]24/25 [mlflow]skinny]]dk]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


#### **IMPORT THE NECESSARY LIBRARIES**

In [12]:

import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import average_precision_score
import os


#### **Setup the mlflow experiment**

In [3]:
mlflow.set_experiment("Assignment_02_SMS_Spam")


2026/03/03 23:13:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/03 23:13:20 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location=('file:///c:/Users/Arnab '
 'Bera/Desktop/CMI/CMI/Applied-ML/AppliedMachineLearning-main/AppliedMachineLearning/Assignment-02/mlruns/1'), creation_time=1771172939288, experiment_id='1', last_update_time=1771172939288, lifecycle_stage='active', name='Assignment_02_SMS_Spam', tags={}, workspace='default'>

#### **Load the dataset**

In [4]:
train_df = pd.read_csv("dataset/train.csv")
val_df = pd.read_csv("dataset/validation.csv")
test_df = pd.read_csv("dataset/test.csv")

X_train = train_df["message"]
y_train = train_df["label"]

X_val = val_df["message"]
y_val = val_df["label"]

X_test = test_df["message"]
y_test = test_df["label"]


#### **Text Vectorization**

In [5]:
vectorizer = TfidfVectorizer(stop_words="english")

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)


In [15]:
os.makedirs("mlruns", exist_ok=True)
mlflow.set_tracking_uri("file://" + os.path.abspath("mlruns"))

mlflow.set_experiment("EVRP_Logistic_Experiments")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/03 23:17:57 INFO mlflow.tracking.fluent: Experiment with name 'EVRP_Logistic_Experiments' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///Users/anirban/SEM-4/AML/AppliedMachineLearning/Assignment-02/mlruns/346204385419735783', creation_time=1772560077411, experiment_id='346204385419735783', last_update_time=1772560077411, lifecycle_stage='active', name='EVRP_Logistic_Experiments', tags={}, workspace='default'>

#### **MODEDL-1: Logistic Regression**

In [16]:
with mlflow.start_run(run_name="Logistic_Regression"):

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_vec, y_train)

    probs = model.predict_proba(X_test_vec)[:, 1]
    aucpr = average_precision_score(y_test, probs)

    mlflow.log_param("model_name", "LogisticRegression")
    
    mlflow.log_metric("AUCPR", aucpr)

    mlflow.sklearn.log_model(model, "model")

    print("Logistic Regression AUCPR:", aucpr)


2026/03/03 23:18:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 23:18:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic Regression AUCPR: 0.9646995083850685


#### **MODEL-2: NB**

In [17]:
with mlflow.start_run(run_name="Naive_Bayes"):

    model = MultinomialNB()
    model.fit(X_train_vec, y_train)

    probs = model.predict_proba(X_test_vec)[:, 1]
    aucpr = average_precision_score(y_test, probs)

    mlflow.log_param("model_name", "MultinomialNB")
    mlflow.log_metric("AUCPR", aucpr)

    mlflow.sklearn.log_model(model, "model")

    print("Naive Bayes AUCPR:", aucpr)


2026/03/03 23:18:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 23:18:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Naive Bayes AUCPR: 0.9786721665112575


#### **MODEL-3: Linear SVM**

In [18]:
with mlflow.start_run(run_name="Linear_SVM"):

    model = LinearSVC()
    model.fit(X_train_vec, y_train)

    scores = model.decision_function(X_test_vec)
    aucpr = average_precision_score(y_test, scores)

    mlflow.log_param("model_name", "LinearSVC")
    mlflow.log_metric("AUCPR", aucpr)

    mlflow.sklearn.log_model(model, "model")

    print("Linear SVM AUCPR:", aucpr)


2026/03/03 23:18:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 23:18:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear SVM AUCPR: 0.9851493315631115


#### **Show all the model**

In [23]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiment_name = "EVRP_Logistic_Experiments"  # change if needed
experiment = client.get_experiment_by_name(experiment_name)

if experiment is None:
    print("Experiment not found!")
else:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.AUCPR DESC"]
    )

    print(f"\nAll Model AUCPR Scores in {experiment_name}:\n")


    for run in runs:
        model_name = run.data.params.get("model_name", "Unknown")
        aucpr = run.data.metrics.get("AUCPR", "Not Logged")
        print(model_name, " : AUCPR:", aucpr)


All Model AUCPR Scores in EVRP_Logistic_Experiments:

LinearSVC  : AUCPR: 0.9851493315631115
MultinomialNB  : AUCPR: 0.9786721665112575
LogisticRegression  : AUCPR: 0.9646995083850685
